In [3]:
import pandas as pd
import numpy as np

def calculate_dispatch_score(file_path):

    df = pd.read_csv("/content/nyc_311_2025_07.csv")

    # 確保時間與郵遞區號格式正確
    df['created_date'] = pd.to_datetime(df['created_date'])
    df['incident_zip'] = df['incident_zip'].fillna(0).astype(int).astype(str)

    # 2. 定義信號類別
    # 正向信號：通常代表有聚會、活動，潛在需求高
    positive_types = [
        'Noise - Residential', 'Loud Music/Party',
        'Noise - Street/Sidewalk', 'Noise - Commercial'
    ]
    # 負向信號：代表交通阻礙、容易被開單、行車效率低
    negative_types = [
        'Blocked Driveway', 'Illegal Parking',
        'Double Parked Vehicle', 'Abandoned Vehicle'
    ]

    # 3. 提取特徵：按郵遞區號聚合
    # 計算正向報案數 (Demand Index)
    demand_counts = df[df['complaint_type'].isin(positive_types)].groupby('incident_zip').size()

    # 計算負向報案數 (Constraint Index)
    constraint_counts = df[df['complaint_type'].isin(negative_types)].groupby('incident_zip').size()

    # 4. 建立評分表
    # 合併數據，缺失值補 0
    score_df = pd.DataFrame({
        'demand_volume': demand_counts,
        'constraint_volume': constraint_counts
    }).fillna(0)


    # 避免分母為 0
    def normalize(series):
        if series.max() == series.min():
            return series * 0
        return (series - series.min()) / (series.max() - series.min())

    score_df['demand_norm'] = normalize(score_df['demand_volume'])
    score_df['constraint_norm'] = normalize(score_df['constraint_volume'])

    # 6. 計算最終建議分數 (Dispatch Score)
    # 權重設定：需求權重 0.7，阻礙權重 0.3
    # 公式：Score = (D * 0.7 - C * 0.3) -> 再縮放到 0-100
    raw_score = (score_df['demand_norm'] * 0.7) - (score_df['constraint_norm'] * 0.3)

    #將結果平移並縮放到0-100
    score_df['final_score'] = ((raw_score - raw_score.min()) / (raw_score.max() - raw_score.min()) * 100).round(2)


    def get_recommendation(s):
        if s >= 80: return "⭐⭐⭐⭐⭐ 強烈建議 (熱點且路順)"
        if s >= 60: return "⭐⭐⭐⭐ 推薦前往"
        if s >= 40: return "⭐⭐ 普通 (穩定區域)"
        return "⚠️避開 (需求低或交通嚴重受阻)"

    score_df['action'] = score_df['final_score'].apply(get_recommendation)

    # 輸出前0名最建議前往的郵遞區號
    return score_df.sort_values(by='final_score', ascending=False)

# 執行分析
result = calculate_dispatch_score('nyc_311_2025_07.csv')

# 顯示前15名熱點
print("紐約市計程車調度建議排行榜")
print(result[['demand_volume', 'constraint_volume', 'final_score', 'action']].head(15))

# 儲存結果
result.to_csv('taxi_dispatch_scores.csv')

紐約市計程車調度建議排行榜
              demand_volume  constraint_volume  final_score  \
incident_zip                                                  
10452                1720.0              400.0       100.00   
10456                1560.0              364.0        92.22   
10457                1458.0              376.0        86.57   
10453                1379.0              378.0        82.34   
10468                1310.0              474.0        76.77   
10029                1065.0              142.0        70.35   
10031                1035.0              292.0        65.78   
10032                1054.0              368.0        65.28   
10467                1108.0              577.0        64.00   
10033                 893.0              180.0        60.46   
10458                 953.0              436.0        58.57   
10460                 884.0              358.0        56.46   
10040                 746.0              149.0        53.28   
11221                 842.0              